In [1]:
# 1138-01  Use 2024 computer rates

In [1]:
import geopandas as gpd
from shapely.geometry import Point, Polygon, LineString
import numpy as np
import pandas as pd
import glob
import re
import matplotlib.pyplot as plt
import os
import pyvista as pv
from pathlib import Path

pd.set_option('display.max_columns', None)


In [2]:
spath = r"C:\Users\cakyol\gpkgs\**.gpkg"
files = [x.replace('\\','/') for x in glob.glob(spath, recursive=True)]

#spath = r"C:\scipts\1138-01\Sonar\PPG2\1993\cynthia\*.gpkg"
#files = files + [x.replace('\\','/') for x in glob.glob(spath, recursive=True)]

print(len(files))
df = pd.DataFrame({'path':files})
df['fname'] = df['path'].apply(lambda x: x.split('/')[-1])
df['orient'] = 'v'
df['orient'] = df['orient'].where(df.fname.apply( lambda x:'ft' not in x), other='h')

# Drop Bad row
df.drop(index=[14], inplace=True)

## well
import re

df['Well_No'] = None

for index, row in df.iterrows():

    fname = row['fname']
    fname_upper = fname.upper()

    # -------------------------
    # HORIZONTAL FILES
    # -------------------------
    if row['orient'] == 'h':

        m = re.search(r'\d{4}', fname)
        if m:
            df.at[index, 'z'] = int(m.group())

        df.at[index, 'Well_No'] = 'all'


    # -------------------------
    # VERTICAL FILES
    # -------------------------
    else:

        # extract azimuths
        m = re.search(r'\d+-\d+', fname)
        if m:
            a0, a1 = m.group().split('-')
            df.at[index, 'azi0'] = int(a0)
            df.at[index, 'azi1'] = int(a1)

        # assign well based on filename
        if 'ALL' in fname_upper:
            df.at[index, 'Well_No'] = 'all'

        elif 'PPG4' in fname_upper:
            df.at[index, 'Well_No'] = 'PPG4'

        elif 'PPG2' in fname_upper:
            df.at[index, 'Well_No'] = 'PPG2'

        else:
            df.at[index, 'Well_No'] = 'UNKNOWN'

25


In [3]:
df

,path,fname,orient,Well_No,azi0,azi1,z
0,C:/Users/cakyol/gpkgs/0-180_PPG2.gpkg,0-180_PPG2.gpkg,v,PPG2,0.0,180.0,NaN
1,C:/Users/cakyol/gpkgs/0-180_PPG4.gpkg,0-180_PPG4.gpkg,v,PPG4,0.0,180.0,NaN
2,C:/Users/cakyol/gpkgs/225-45_PPG2.gpkg,225-45_PPG2.gpkg,v,PPG2,225.0,45.0,NaN
3,C:/Users/cakyol/gpkgs/225-45_PPG4.gpkg,225-45_PPG4.gpkg,v,PPG4,225.0,45.0,NaN
4,C:/Users/cakyol/gpkgs/2600ft.gpkg,2600ft.gpkg,h,all,NaN,NaN,2600.0
5,C:/Users/cakyol/gpkgs/270-90_PPG2.gpkg,270-90_PPG2.gpkg,v,PPG2,270.0,90.0,NaN
6,C:/Users/cakyol/gpkgs/270-90_PPG4_1.gpkg,270-90_PPG4_1.gpkg,v,PPG4,270.0,90.0,NaN
7,C:/Users/cakyol/gpkgs/270-90_PPG4_2.gpkg,270-90_PPG4_2.gpkg,v,PPG4,270.0,90.0,NaN
8,C:/Users/cakyol/gpkgs/2700ft.gpkg,2700ft.gpkg,h,all,NaN,NaN,2700.0
9,C:/Users/cakyol/gpkgs/277-97_PPG2.gpkg,277-97_PPG2.gpkg,v,PPG2,277.0,97.0,NaN


In [4]:
## dX, dY
# ppg 2 
2624245.599498,643042.320207

# ppg 4
2624685.606284,642633.314476

#horizontal offsets
#ppg2 dy:155, dx:-290
#ppg4 dy:-120, dx:180

(2624685.606284, 642633.314476)

In [5]:
# =========================
# USER INPUTS
# =========================
CRS_OUT = "EPSG:3452"

# Main 0-reference point for the cavern
CAVERN_CENTER = {
    "X0": 2624465.602891,
    "Y0": 642837.817341
}

# Wellhead offsets from cavern center
# dx = East-West difference
# dy = North-South difference
WELL_OFFSETS_FROM_CAVERN_CENTER = {
    "PPG2": {
        "dx": -300,
        "dy":  155
    },
    "PPG4": {
        "dx":  180,
        "dy": -120
    }
}

# Optional: rebuild WELLHEADS automatically from center + offsets
WELLHEADS = {
    well: {
        "Xwh": CAVERN_CENTER["X0"] + off["dx"],
        "Ywh": CAVERN_CENTER["Y0"] + off["dy"]
    }
    for well, off in WELL_OFFSETS_FROM_CAVERN_CENTER.items()
}

SCALE_LOCAL_TO_MAP = 1.0
STEP = 10.0
VERT_AZ_MODE = "mean"
VERTICAL_X_IS_YLOCAL = True

OUT_GPKG = "PPG_1993_referenced_pointclouds2.gpkg"
OUT_LAYER = "sonar_points"

In [6]:
# =========================
# Helper functions
# =========================
def circular_mean_deg(a, b):
    a = np.deg2rad(a); b = np.deg2rad(b)
    x = np.cos(a) + np.cos(b)
    y = np.sin(a) + np.sin(b)
    return (np.rad2deg(np.arctan2(y, x)) + 360) % 360

def get_vertical_azimuths(row):
    a0, a1 = row.get("azi0", np.nan), row.get("azi1", np.nan)
    if pd.isna(a0) and pd.isna(a1):
        return []
    if VERT_AZ_MODE == "azi0":
        return [float(a0)]
    if VERT_AZ_MODE == "azi1":
        return [float(a1)]
    if VERT_AZ_MODE == "mean":
        return [float(circular_mean_deg(float(a0), float(a1)))]
    if VERT_AZ_MODE == "both":
        out = []
        if not pd.isna(a0): out.append(float(a0))
        if not pd.isna(a1): out.append(float(a1))
        return out
    raise ValueError("VERT_AZ_MODE must be one of: azi0, azi1, mean, both")

def sample_geom_xy(geom, step=STEP):
    """Sample boundary points from Polygon/LineString into Nx2 array."""
    if geom is None or geom.is_empty:
        return np.empty((0, 2), dtype=float)

    if isinstance(geom, Polygon):
        line = geom.exterior
    elif isinstance(geom, LineString):
        line = geom
    else:
        b = geom.boundary
        if isinstance(b, LineString):
            line = b
        else:
            return np.empty((0, 2), dtype=float)

    L = line.length
    if L <= 0:
        return np.empty((0, 2), dtype=float)

    n = max(2, int(np.ceil(L / step)))
    dists = np.linspace(0, L, n)
    pts = [line.interpolate(d) for d in dists]
    return np.array([[p.x, p.y] for p in pts], dtype=float)

def cavern_center_from_wells():
    centers = {
        well: (
            CAVERN_CENTER["X0"],
            CAVERN_CENTER["Y0"]
        )
        for well in WELL_OFFSETS_FROM_CAVERN_CENTER.keys()
    }
    return (CAVERN_CENTER["X0"], CAVERN_CENTER["Y0"]), centers

def wellhead_from_center(well):
    Xw = CAVERN_CENTER["X0"] + WELL_OFFSETS_FROM_CAVERN_CENTER[well]["dx"]
    Yw = CAVERN_CENTER["Y0"] + WELL_OFFSETS_FROM_CAVERN_CENTER[well]["dy"]
    return Xw, Yw


In [7]:
def section_origin_from_well(well, az_deg, offset):
    Xw, Yw = wellhead_from_center(well)
    az = np.deg2rad(float(az_deg))
    X0 = Xw - offset * np.sin(az)
    Y0 = Yw - offset * np.cos(az)
    return X0, Y0

Xsec_from_ppg2, Ysec_from_ppg2 = section_origin_from_well("PPG2", 300, -330)
Xsec_from_ppg4, Ysec_from_ppg4 = section_origin_from_well("PPG4", 300, 210)

print("300-120 origin from PPG2:", Xsec_from_ppg2, Ysec_from_ppg2)
print("300-120 origin from PPG4:", Xsec_from_ppg4, Ysec_from_ppg4)
print("Difference:", Xsec_from_ppg2 - Xsec_from_ppg4, Ysec_from_ppg4 - Ysec_from_ppg4)

Xsec_300120 = 0.5 * (Xsec_from_ppg2 + Xsec_from_ppg4)
Ysec_300120 = 0.5 * (Ysec_from_ppg2 + Ysec_from_ppg4)

print("Using 300-120 section origin:", Xsec_300120, Ysec_300120)
print("Difference:", Xsec_from_ppg2 - Xsec_from_ppg4, Ysec_from_ppg2 - Ysec_from_ppg4)

300-120 origin from PPG2: 2623879.8145077513 643157.817341
300-120 origin from PPG4: 2624827.468225795 642612.817341
Difference: -947.6537180435844 0.0
Using 300-120 section origin: 2624353.6413667733 642885.317341
Difference: -947.6537180435844 545.0


In [8]:
# Check cavern center computed from each well
for w in ["PPG2", "PPG4"]:
    Xw, Yw = wellhead_from_center(w)
    print(w, "wellhead:", Xw, Yw)

print("Cavern center:", CAVERN_CENTER["X0"], CAVERN_CENTER["Y0"])

Xc2 = WELLHEADS["PPG2"]["Xwh"] - WELL_OFFSETS_FROM_CAVERN_CENTER["PPG2"]["dx"]
Yc2 = WELLHEADS["PPG2"]["Ywh"] - WELL_OFFSETS_FROM_CAVERN_CENTER["PPG2"]["dy"]
Xc4 = WELLHEADS["PPG4"]["Xwh"] - WELL_OFFSETS_FROM_CAVERN_CENTER["PPG4"]["dx"]
Yc4 = WELLHEADS["PPG4"]["Ywh"] - WELL_OFFSETS_FROM_CAVERN_CENTER["PPG4"]["dy"]

print("Center diff PPG2-PPG4:", (Xc2-Xc4), (Yc2-Yc4))

PPG2 wellhead: 2624165.602891 642992.817341
PPG4 wellhead: 2624645.602891 642717.817341
Cavern center: 2624465.602891 642837.817341
Center diff PPG2-PPG4: 0.0 0.0


In [9]:
# =========================
# Build referenced point cloud + metadata
# =========================


(Xcav, Ycav), centers = cavern_center_from_wells()
print("Cavern center from wells (per-well):", centers)
print("Using cavern center (average):", (Xcav, Ycav))


Xsec = 0.5 * (WELLHEADS["PPG2"]["Xwh"] + WELLHEADS["PPG4"]["Xwh"])
Ysec = 0.5 * (WELLHEADS["PPG2"]["Ywh"] + WELLHEADS["PPG4"]["Ywh"])

print("Section midpoint for ALL verticals:", (Xsec, Ysec))

# ---------------------------------
# Fit cavern center for 300-120_all
# ---------------------------------
az_300120 = np.deg2rad(120.0)

# distances along section axis from cavern center
s_ppg2 = -330.0
s_ppg4 = 290.0

Xcav2 = WELLHEADS["PPG2"]["Xwh"] - s_ppg2 * np.sin(az_300120)
Ycav2 = WELLHEADS["PPG2"]["Ywh"] - s_ppg2 * np.cos(az_300120)

Xcav4 = WELLHEADS["PPG4"]["Xwh"] - s_ppg4 * np.sin(az_300120)
Ycav4 = WELLHEADS["PPG4"]["Ywh"] - s_ppg4 * np.cos(az_300120)

Xcav_fit = 0.5 * (Xcav2 + Xcav4)
Ycav_fit = 0.5 * (Ycav2 + Ycav4)

print("Fitted cavern center for 300-120:", (Xcav_fit, Ycav_fit))

records = []

for _, row in df.iterrows():

    orient = row["orient"]
    fname  = row["fname"]
    path   = row["path"]
    wellno = str(row.get("Well_No", "all")).lower()

    gdf = gpd.read_file(path)
    if gdf.empty:
        continue

    # -------------------------
    # HORIZONTAL: cavern-centered
    # -------------------------
    if orient == "h":

        depth = row.get("z", np.nan)
        if pd.isna(depth):
            continue

        for geom in gdf.geometry:
            xy = sample_geom_xy(geom, step=STEP)
            if xy.size == 0:
                continue

            x_local = xy[:, 0]
            y_local = xy[:, 1]

            X = Xcav + x_local
            Y = Ycav + y_local
            Z = np.full_like(X, -float(depth), dtype=float)

            for xi, yi, zi in zip(X, Y, Z):
                records.append({
                    "fname": fname,
                    "path": path,
                    "orient": "h",
                    "depth": float(depth),
                    "azi0": np.nan,
                    "azi1": np.nan,
                    "az_used": np.nan,
                    "Well_No": wellno,
                    "geometry": Point(float(xi), float(yi), float(zi)),
                })

    # -------------------------
    # VERTICAL
    # -------------------------
    elif orient == "v":

        az_list = get_vertical_azimuths(row)
        if not az_list:
            continue

        a0 = row.get("azi0", np.nan)
        a1 = row.get("azi1", np.nan)

        # choose anchor
        if wellno == "ppg2":
            X0 = WELLHEADS["PPG2"]["Xwh"]
            Y0 = WELLHEADS["PPG2"]["Ywh"]

        elif wellno == "ppg4":
            X0 = WELLHEADS["PPG4"]["Xwh"]
            Y0 = WELLHEADS["PPG4"]["Ywh"]

        elif wellno == "all":
            if "300-120" in fname:
                X0 = Xcav_fit
                Y0 = Ycav_fit
            else:
                X0 = Xsec
                Y0 = Ysec

        else:
            print(f"Skipping {fname}: unknown Well_No = {wellno}")
            continue

        for az_deg in az_list:

#ROTATING WITH RELATIVE AZIMUTH************************************************
#300-120_all
            if "300-120" in fname:
                az_use = 120.0
#277-97_PPG2
            elif "277-97" in fname and wellno == "ppg2":
                # rotate relative to original azimuth, keep wellhead fixed
                az_use = (float(az_deg) + 90.0) % 360
                print("ROTATING 277-97_PPG2:", fname, "old:", az_deg, "new:", az_use)
#270-90_PPG4_2
            elif fname == "270-90_PPG4_2.gpkg" and wellno == "ppg4":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
#270-90_PPG4_1
            elif fname == "270-90_PPG4_1.gpkg" and wellno == "ppg4":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
#315-135_PPG4
            elif fname == "315-135_PPG4.gpkg" and wellno == "ppg4":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
#315-135_PPG2
            elif fname == "315-135_PPG2.gpkg" and wellno == "ppg2":
                az_use = (float(az_deg) - 90.0) % 360
                print("ROTATING:", fname, "old:", az_deg, "new:", az_use)
            else:
                az_use = float(az_deg)

            az = np.deg2rad(az_use)
#*****************************************************************************
#SHIFTS

    
            x_shift = 0.0
            y_shift = 0.0

            for geom in gdf.geometry:
                xy = sample_geom_xy(geom, step=STEP)
                if xy.size == 0:
                    continue

                if VERTICAL_X_IS_YLOCAL:
                    y_local = xy[:, 0]
                    z_local = xy[:, 1]
                else:
                    y_local = xy[:, 1]
                    z_local = xy[:, 0]

                dX = y_local * np.sin(az)
                dY = y_local * np.cos(az)

                X = X0 + dX + x_shift
                Y = Y0 + dY + y_shift
                Z = z_local.astype(float)

                for xi, yi, zi in zip(X, Y, Z):
                    records.append({
                        "fname": fname,
                        "path": path,
                        "orient": "v",
                        "depth": np.nan,
                        "azi0": float(a0) if not pd.isna(a0) else np.nan,
                        "azi1": float(a1) if not pd.isna(a1) else np.nan,
                        "az_used": az_use,
                        "Well_No": wellno,
                        "geometry": Point(float(xi), float(yi), float(zi)),
                    })

gdf_out = gpd.GeoDataFrame(records, geometry="geometry", crs=CRS_OUT)
print("Total referenced points:", len(gdf_out))

Cavern center from wells (per-well): {'PPG2': (2624465.602891, 642837.817341), 'PPG4': (2624465.602891, 642837.817341)}
Using cavern center (average): (2624465.602891, 642837.817341)
Section midpoint for ALL verticals: (2624405.602891, 642855.317341)
Fitted cavern center for 300-120: (2624422.923399076, 642845.317341)
ROTATING: 270-90_PPG4_1.gpkg old: 180.0 new: 90.0
ROTATING: 270-90_PPG4_2.gpkg old: 180.0 new: 90.0
ROTATING 277-97_PPG2: 277-97_PPG2.gpkg old: 19.17900802581073 new: 109.17900802581073
ROTATING: 315-135_PPG2.gpkg old: 225.0 new: 135.0
ROTATING: 315-135_PPG4.gpkg old: 225.0 new: 135.0
Total referenced points: 6548


In [10]:

def shift_point_3d(p, dx=0.0, dy=0.0, dz=0.0):
    return Point(p.x + dx, p.y + dy, p.z + dz)

def rotate_point_xy_around_anchor(p, angle_deg, x0, y0):
    """
    Rotate a 3D point in the XY plane around anchor (x0, y0).
    Z stays unchanged.
    Positive angle = counterclockwise.
    """
    ang = np.deg2rad(angle_deg)

    dx = p.x - x0
    dy = p.y - y0

    xr = dx * np.cos(ang) - dy * np.sin(ang)
    yr = dx * np.sin(ang) + dy * np.cos(ang)

    return Point(x0 + xr, y0 + yr, p.z)

def transform_gdf_points(
    gdf,
    mask,
    angle_deg=0.0,
    anchor_x=None,
    anchor_y=None,
    dx=0.0,
    dy=0.0,
    dz=0.0
):
    gdf2 = gdf.copy()

    def _transform(p):
        p2 = p

        if angle_deg != 0.0:
            if anchor_x is None or anchor_y is None:
                raise ValueError("anchor_x and anchor_y are required for rotation.")
            p2 = rotate_point_xy_around_anchor(p2, angle_deg, anchor_x, anchor_y)

        if dx != 0.0 or dy != 0.0 or dz != 0.0:
            p2 = shift_point_3d(p2, dx=dx, dy=dy, dz=dz)

        return p2

    gdf2.loc[mask, "geometry"] = gdf2.loc[mask, "geometry"].apply(_transform)
    return gdf2

In [11]:
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt

plotter = pv.Plotter()

files = gdf_out["fname"].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(files)))

for i, fname in enumerate(files):

    sub = gdf_out[gdf_out["fname"] == fname]

    coords = np.array([
        (g.x, g.y, g.z)
        for g in sub.geometry
    ])

    cloud = pv.PolyData(coords)

    # Highlight important sections
    if "315-135_PPG4" in fname:
        plotter.add_mesh(
            cloud,
            color="red",
            render_points_as_spheres=True,
            point_size=12,
            label=fname
        )

    else:
        plotter.add_mesh(
            cloud,
            color=colors[i][:3],
            render_points_as_spheres=True,
            point_size=5,
            opacity=0.5,
            label=fname
        )

pv.global_theme.font.size = 58

plotter.add_legend(
    loc="upper right",
    size=(0.15, 0.15),
    bcolor="white"
)

plotter.show()

Widget(value='<iframe src="http://localhost:60027/index.html?ui=P_0x1fa7fe44d50_0&reconnect=auto" class="pyvis…

In [12]:
# -------------------------------
# EDIT ONE VERTICAL SECTION AFTER BUILD
# -------------------------------

#Bullk shift of 300-120_all.gpkg

mask = (
    (gdf_out["orient"] == "v") &
    (gdf_out["fname"].str.contains("300-120", case=False, na=False))
)

gdf_edit = transform_gdf_points(
    gdf_out,
    mask=mask,
    angle_deg=0.0,         # try 2, -2, 5, etc.
    anchor_x=Xcav_fit,     # or Xsec / PPG2 / PPG4 wellhead
    anchor_y=Ycav_fit,
    dx=30.0,
    dy=-28.0,
    dz=0.0
)

#Rotating 315-135_PPG4 (includes 1981 sonar)
# second fix
mask = (
    (gdf_edit["orient"] == "v") &
    (gdf_edit["fname"].str.contains("277-97_PPG2", case=False, na=False)) &
    (gdf_edit["Well_No"] == "ppg2")
)

gdf_edit = transform_gdf_points(
    gdf_edit,
    mask=mask,
    angle_deg=0.0,
    anchor_x=WELLHEADS["PPG2"]["Xwh"],
    anchor_y=WELLHEADS["PPG2"]["Ywh"],
    dx=280.0,
    dy=-125.0,
    dz=0.0
)
# -------------------------------
# SHIFT 2700ft HORIZONTAL TO 2900ft
# -------------------------------

mask = (
    (gdf_edit["orient"] == "h") &
    (gdf_edit["fname"].str.contains("2700", case=False, na=False))
)

gdf_edit = transform_gdf_points(
    gdf_edit,
    mask=mask,
    angle_deg=0.0,   # no rotation
    anchor_x=0,
    anchor_y=0,
    dx=0.0,
    dy=0.0,
    dz=-50.0        # move from 2700 to 2900
)


In [13]:


plotter = pv.Plotter()

files = gdf_edit["fname"].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(files)))

for i, fname in enumerate(files):

    sub = gdf_edit[gdf_edit["fname"] == fname]

    coords = np.array([
        (g.x, g.y, g.z)
        for g in sub.geometry
    ])

    cloud = pv.PolyData(coords)

    if "315-135_PPG2" in fname:
        plotter.add_mesh(
            cloud,
            color="red",
            render_points_as_spheres=True,
            point_size=12,
            label=fname
        )
    else:
        plotter.add_mesh(
            cloud,
            color=colors[i][:3],
            render_points_as_spheres=True,
            point_size=5,
            opacity=0.5,
            label=fname
        )

pv.global_theme.font.size = 58

plotter.add_legend(
    loc="upper right",
    size=(0.15, 0.15),
    bcolor="white"
)

plotter.show()

Widget(value='<iframe src="http://localhost:60027/index.html?ui=P_0x1fa06685190_1&reconnect=auto" class="pyvis…

In [ ]:

# =========================================================
# SYNTHETIC MESH — TWO SEPARATE CAVERNS (PPG2 + PPG4)
#
# Each cavern meshed independently using its wellhead as
# the polar coordinate center.
# Method: 2D scattered interpolation r = f(azi, Z)
#   - For each (azi_bin, Z_bin) keep MAX r → outer wall
#   - scipy.griddata linear interpolation on regular grid
#   - Light Gaussian smoothing to remove jagged steps
# =========================================================

import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
import pyvista as pv

# ----------------------------------------------------------
# PARAMETERS
# ----------------------------------------------------------
Z_STEP       = 2.5    # vertical spacing between rings (ft) — 2x more rings closes gaps
AZI_STEP     = 2      # degrees between azimuths
R_MAX        = 800.0  # hard cap — only removes true outliers (ft)
SMOOTH_SIGMA = [2.0, 0.3]  # [z_axis, azi_axis] — heavier Z smoothing closes inter-ring gaps
N_RING       = 360 // AZI_STEP   # = 180

# bin sizes for max-r aggregation (match output resolution)
AZI_BIN = 2.0   # degrees  (same as AZI_STEP)
Z_BIN   = 5.0   # ft

WELLS = {
    "PPG2": {"X0": WELLHEADS["PPG2"]["Xwh"], "Y0": WELLHEADS["PPG2"]["Ywh"], "color": "mediumpurple"},
    "PPG4": {"X0": WELLHEADS["PPG4"]["Xwh"], "Y0": WELLHEADS["PPG4"]["Ywh"], "color": "sienna"},
}

# ----------------------------------------------------------
# HELPER — build mesh for one well
# ----------------------------------------------------------
def build_cavern_mesh(pts_gdf, X0, Y0, z_step, azi_step,
                      r_max, azi_bin, z_bin, smooth_sigma, n_ring):
    gdf = pts_gdf.copy()
    gdf["x"] = gdf.geometry.x
    gdf["y"] = gdf.geometry.y
    gdf["z"] = gdf.geometry.apply(lambda g: g.z if g.has_z else np.nan)
    gdf = gdf.dropna(subset=["x", "y", "z"]).reset_index(drop=True)

    gdf["r"]   = np.sqrt((gdf["x"] - X0)**2 + (gdf["y"] - Y0)**2)
    gdf["azi"] = (np.degrees(np.arctan2(gdf["x"] - X0,
                                        gdf["y"] - Y0)) + 360) % 360

    # hard cap — only removes true outlier reflections
    gdf = gdf[gdf["r"] <= r_max].copy()

    if len(gdf) < 20:
        print("  Not enough points after filtering.")
        return None

    # --- keep MAX r per (azi_bin, z_bin) → outer cavern wall ---
    gdf["azi_bin"] = (gdf["azi"] // azi_bin) * azi_bin
    gdf["z_bin"]   = (gdf["z"]   // z_bin)   * z_bin
    wall = (gdf.groupby(["azi_bin", "z_bin"])["r"]
              .max()
              .reset_index()
              .rename(columns={"azi_bin": "azi", "z_bin": "z"}))

    # Per-Z-ring IQR outlier removal — strips spurious far reflections
    # that survive the global R_MAX cap but are outliers within their Z slice
    clean_rings = []
    for z_val, grp in wall.groupby("z"):
        q1, q3 = grp["r"].quantile(0.25), grp["r"].quantile(0.75)
        iqr = q3 - q1
        upper = q3 + 3.0 * iqr
        clean_rings.append(grp[grp["r"] <= upper])
    wall = pd.concat(clean_rings, ignore_index=True)

    print(f"  {len(gdf)} pts → {len(wall)} wall samples  "
          f"Z: {wall['z'].min():.0f}→{wall['z'].max():.0f}  "
          f"r: {wall['r'].min():.0f}→{wall['r'].max():.0f}")

    # 2D scattered interpolation with circular wrap-around
    azi_pts = wall["azi"].values
    z_pts   = wall["z"].values
    r_pts   = wall["r"].values

    azi_aug = np.concatenate([azi_pts - 360, azi_pts, azi_pts + 360])
    z_aug   = np.tile(z_pts, 3)
    r_aug   = np.tile(r_pts, 3)

    azis   = np.arange(0, 360, azi_step)
    # Only keep Z levels with sufficient azimuthal coverage (>=50%)
    # Fall back to progressively lower thresholds if no levels qualify
    for thresh in [0.50, 0.30, 0.10, 0.0]:
        min_azi_count = max(1, int(thresh * (360.0 / azi_bin)))
        z_coverage = wall.groupby("z")["azi"].count()
        valid_z = z_coverage[z_coverage >= min_azi_count].index
        if len(valid_z) >= 2:
            break
    if len(valid_z) < 2:
        print("  Not enough Z levels with coverage — skipping.")
        return None
    z_lo, z_hi = valid_z.min(), valid_z.max()
    z_grid = np.arange(z_lo, z_hi + z_step, z_step)
    AZI_G, Z_G = np.meshgrid(azis, z_grid)

    r_grid = griddata(
        np.column_stack([azi_aug, z_aug]),
        r_aug,
        np.column_stack([AZI_G.ravel(), Z_G.ravel()]),
        method="linear"
    ).reshape(AZI_G.shape)

    # Fill NaNs:
    # - interpolate fills interior gaps linearly along Z per azimuth column
    # - ffill/bfill extend to leading/trailing edge NaNs in each column
    # - second axis=1 pass fills any azimuth columns with no data at all
    nan_mask = np.isnan(r_grid)
    if nan_mask.any():
        r_df = pd.DataFrame(r_grid)   # shape (n_z, n_azi)
        r_df = (r_df.interpolate(method="linear", axis=0)
                    .ffill(axis=0).bfill(axis=0))   # fill edge NaNs along Z
        r_df = (r_df.interpolate(method="linear", axis=1)
                    .ffill(axis=1).bfill(axis=1))   # fill empty azi columns
        r_grid = r_df.values

    # light Gaussian smoothing
    r_grid = gaussian_filter(r_grid, sigma=smooth_sigma)

    # build rings
    azi_rad = np.deg2rad(azis)
    rings = []
    for i, z in enumerate(z_grid):
        r_row = r_grid[i]
        rings.append(np.column_stack([
            X0 + r_row * np.sin(azi_rad),
            Y0 + r_row * np.cos(azi_rad),
            np.full(n_ring, z)
        ]))

    all_pts = np.vstack(rings)
    faces   = []
    base    = [i * n_ring for i in range(len(rings))]
    for i in range(len(rings) - 1):
        b0, b1 = base[i], base[i + 1]
        for j in range(n_ring):
            jn = (j + 1) % n_ring
            faces += [[3, b0+j, b1+j, b0+jn],
                      [3, b0+jn, b1+j, b1+jn]]

    # bottom cap (last ring = lowest z after sort)
    for cap_ring_idx, cap_base in [(0, base[0]), (-1, base[-1])]:
        ring_pts = rings[cap_ring_idx]
        cx = ring_pts[:, 0].mean()
        cy = ring_pts[:, 1].mean()
        cz = ring_pts[0, 2]
        cap_center_idx = len(all_pts)
        all_pts = np.vstack([all_pts, [[cx, cy, cz]]])
        for j in range(n_ring):
            jn = (j + 1) % n_ring
            faces.append([3, cap_center_idx, cap_base + j, cap_base + jn])

    mesh = pv.PolyData(all_pts, np.hstack(faces))
    mesh = mesh.compute_normals(auto_orient_normals=True)
    print(f"  → {mesh.n_points} pts, {mesh.n_cells} tri, {len(rings)} rings")
    return mesh

# ----------------------------------------------------------
# STEP 1 — split "all" points to nearest well
# ----------------------------------------------------------
gdf_all = gdf_edit.copy()
gdf_all["x"] = gdf_all.geometry.x
gdf_all["y"] = gdf_all.geometry.y

mask_all = gdf_all["Well_No"].str.lower() == "all"
if mask_all.any():
    pts_all = gdf_all[mask_all]
    X2, Y2 = WELLHEADS["PPG2"]["Xwh"], WELLHEADS["PPG2"]["Ywh"]
    X4, Y4 = WELLHEADS["PPG4"]["Xwh"], WELLHEADS["PPG4"]["Ywh"]
    d2 = np.sqrt((pts_all["x"] - X2)**2 + (pts_all["y"] - Y2)**2)
    d4 = np.sqrt((pts_all["x"] - X4)**2 + (pts_all["y"] - Y4)**2)
    gdf_all.loc[mask_all, "Well_No"] = np.where(d2 <= d4, "ppg2", "ppg4")

# ----------------------------------------------------------
# STEP 2 — build one mesh per well
# ----------------------------------------------------------
meshes = {}
for well, cfg in WELLS.items():
    pts = gdf_all[gdf_all["Well_No"].str.lower() == well.lower()]
    print(f"\n{well}  ({len(pts)} pts):")
    m = build_cavern_mesh(pts, cfg["X0"], cfg["Y0"],
                          Z_STEP, AZI_STEP, R_MAX,
                          AZI_BIN, Z_BIN, SMOOTH_SIGMA, N_RING)
    if m is not None:
        meshes[well] = m

print(f"\nDone — meshes built: {list(meshes.keys())}")


In [ ]:

# =========================================================
# VISUALISE — PPG2 (purple) + PPG4 (brown) + real points
# =========================================================

pl = pv.Plotter()

for well, cfg in WELLS.items():
    if well in meshes:
        pl.add_mesh(meshes[well],
                    color=cfg["color"],
                    opacity=0.7,
                    smooth_shading=True,
                    show_edges=False,
                    label=f"{well} mesh")

# synthetic mesh vertices (ring grid points)
for well, cfg in WELLS.items():
    if well in meshes:
        syn_pts = meshes[well].points
        pl.add_mesh(pv.PolyData(syn_pts),
                    color=cfg["color"],
                    point_size=2,
                    render_points_as_spheres=True,
                    opacity=0.5,
                    label=f"{well} synthetic pts")

real_pts = np.array([(g.x, g.y, g.z) for g in gdf_edit.geometry])
pl.add_mesh(pv.PolyData(real_pts),
            color="red",
            point_size=3,
            render_points_as_spheres=True,
            label="Real points")

pl.add_legend()
pl.add_axes()
pl.show()
